In [1]:
print('hi')

hi


In [6]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Step 1: No missing values in "feature_1" ──
print(f"Missing: {df["feature_1"].isnull().sum()}")  # Should print 0

# ── Step 2: Check for outliers using IQR ──
Q1 = df["feature_1"].quantile(0.25)
Q3 = df["feature_1"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df["feature_1"][(df["feature_1"] < lower_bound) | (df["feature_1"] > upper_bound)]
print(f"Outliers in feature_1: {len(outliers)}")

# ── Step 3: Descriptive statistics ──
print(df["feature_1"].describe())

Loaded dataset: 1000 rows × 11 columns
Missing: 0
Outliers in feature_1: 0
count    1000.000000
mean        0.045702
std         1.148193
min        -1.999262
25%        -0.933481
50%         0.112520
75%         1.029695
max         1.999133
Name: feature_1, dtype: float64


In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



In [3]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.ensemble import HistGradientBoostingClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", HistGradientBoostingClassifier(max_iter=60, max_depth=6, random_state=42)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'hist_gradient_boosting_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: hist_gradient_boosting_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [7]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError("Missing columns: " + str(missing_columns))

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

if len(df) < 10:
    raise ValueError("At least 10 target-complete rows are required.")

X = df[feature_columns]
y = df[target].copy()


from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

if len(target_encoder.classes_) < 2:
    raise ValueError("Classification needs at least two classes.")

class_counts = pd.Series(y).value_counts()
if class_counts.min() < 2:
    raise ValueError(
        "Each class needs at least two rows for stratified splitting."
    )


# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: xgboost_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [9]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.tree import DecisionTreeClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=10, random_state=42)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'decision_tree')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: decision_tree
Training rows: 800
Test rows: 200
Accuracy: 0.925
Balanced accuracy: 0.9237
Classification report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       126
           1       0.88      0.92      0.90        74

    accuracy                           0.93       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.93      0.93      0.93       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [10]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: xgboost_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [11]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.ensemble import GradientBoostingClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(n_estimators=60, random_state=42)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'gradient_boosting_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: gradient_boosting_clf
Training rows: 800
Test rows: 200
Accuracy: 0.975
Balanced accuracy: 0.9718
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       126
           1       0.97      0.96      0.97        74

    accuracy                           0.97       200
   macro avg       0.97      0.97      0.97       200
weighted avg       0.97      0.97      0.97       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError("Missing columns: " + str(missing_columns))

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

if len(df) < 10:
    raise ValueError("At least 10 target-complete rows are required.")

X = df[feature_columns]
y = df[target].copy()


from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

if len(target_encoder.classes_) < 2:
    raise ValueError("Classification needs at least two classes.")

class_counts = pd.Series(y).value_counts()
if class_counts.min() < 2:
    raise ValueError(
        "Each class needs at least two rows for stratified splitting."
    )


# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



In [13]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.tree import DecisionTreeClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError("Missing columns: " + str(missing_columns))

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

if len(df) < 10:
    raise ValueError("At least 10 target-complete rows are required.")

X = df[feature_columns]
y = df[target].copy()


from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

if len(target_encoder.classes_) < 2:
    raise ValueError("Classification needs at least two classes.")

class_counts = pd.Series(y).value_counts()
if class_counts.min() < 2:
    raise ValueError(
        "Each class needs at least two rows for stratified splitting."
    )


# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
# ModelMind: Tuned hyperparameters applied: max_depth=3, min_samples_leaf=4
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42, min_samples_leaf=4)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'decision_tree')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10]



_IncompleteInputError: incomplete input (3571057078.py, line 135)

In [ ]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Step 1: No missing values in "feature_1" ──
print(f"Missing: {df["feature_1"].isnull().sum()}")  # Should print 0

# ── Step 2: Check for outliers using IQR ──
Q1 = df["feature_1"].quantile(0.25)
Q3 = df["feature_1"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df["feature_1"][(df["feature_1"] < lower_bound) | (df["feature_1"] > upper_bound)]
print(f"Outliers in feature_1: {len(outliers)}")

# ── Step 3: Descriptive statistics ──
print(df["feature_1"].describe())

In [14]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Step 1: No missing values in "feature_1" ──
print(f"Missing: {df["feature_1"].isnull().sum()}")  # Should print 0

# ── Step 2: Check for outliers using IQR ──
Q1 = df["feature_1"].quantile(0.25)
Q3 = df["feature_1"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df["feature_1"][(df["feature_1"] < lower_bound) | (df["feature_1"] > upper_bound)]
print(f"Outliers in feature_1: {len(outliers)}")

# ── Step 3: Descriptive statistics ──
print(df["feature_1"].describe())

Loaded dataset: 1000 rows × 11 columns
Missing: 0
Outliers in feature_1: 0
count    1000.000000
mean        0.045702
std         1.148193
min        -1.999262
25%        -0.933481
50%         0.112520
75%         1.029695
max         1.999133
Name: feature_1, dtype: float64


In [16]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: xgboost_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [17]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

X = df[["feature_1"]].copy()
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# "feature_1" has no missing values
numeric_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, ["feature_1"]),
], remainder='passthrough')

# Fit on training data only!
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

Loaded dataset: 1000 rows × 11 columns


In [19]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10]



_IncompleteInputError: incomplete input (1189152057.py, line 119)

In [20]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: xgboost_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [23]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.tree import DecisionTreeClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=10, random_state=42)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'decision_tree')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: decision_tree
Training rows: 800
Test rows: 200
Accuracy: 0.925
Balanced accuracy: 0.9237
Classification report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       126
           1       0.88      0.92      0.90        74

    accuracy                           0.93       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.93      0.93      0.93       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [24]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# "feature_1" has no missing values — nothing to fill!
print(f"Missing: {df["feature_1"].isnull().sum()}")  # Should print 0

Loaded dataset: 1000 rows × 11 columns
Missing: 0


In [22]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv')

target = 'target'
numeric_features = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']
categorical_features = []
feature_columns = numeric_features + categorical_features

required = feature_columns + [target]
missing_columns = [c for c in required if c not in df.columns]

# 2. Remove only observations with unknown targets
df = df.dropna(subset=[target]).copy()

X = df[feature_columns]
y = df[target].copy()

from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

class_counts = pd.Series(y).value_counts()

# 3. Split BEFORE fitting preprocessing statistics
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. Create preprocessing steps
transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

# 5. Fit preprocessing and model on training rows only
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, tree_method='hist', eval_metric='logloss', n_jobs=1, random_state=42, verbosity=0)),
])

pipeline.fit(X_train, y_train)

# 6. Evaluate on held-out test rows
predictions = pipeline.predict(X_test)

print("Model:", 'xgboost_clf')
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, predictions), 4),
)

print("Classification report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

decoded_predictions = target_encoder.inverse_transform(
    predictions.astype(int)
)
print("Predictions:", decoded_predictions[:10])



Model: xgboost_clf
Training rows: 800
Test rows: 200
Accuracy: 0.98
Balanced accuracy: 0.9758
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       126
           1       0.99      0.96      0.97        74

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200

Predictions: ['0' '0' '1' '0' '0' '0' '1' '0' '0' '0']


In [21]:
# Recommended preprocessing workflow for the selected skill level
import pandas as pd

df = pd.read_csv("/Users/harshit9793/ML plateform/Backup2/ShowCase/classification_tuning_1000_rows (1).csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# "feature_1" has no missing values — nothing to fill!
print(f"Missing: {df["feature_1"].isnull().sum()}")  # Should print 0

Loaded dataset: 1000 rows × 11 columns
Missing: 0
